# Simple Neo4j KG Demo

This notebook creates a tiny knowledge graph for `LLM`, `RAG`, and `Knowledge Graph` concepts, then queries it with Cypher. The final section optionally uses `Text2Cypher` if `OPENAI_API_KEY` is available.


## 1. Connect to Neo4j

Required environment variables:

- `NEO4J_URI`
- `NEO4J_USERNAME`
- `NEO4J_PASSWORD`

Optional for the last section:

- `OPENAI_API_KEY`


In [ ]:
import os
from pprint import pprint

from utils import neo4j_driver
from schema_utils import get_schema

missing = [
    name
    for name in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
    if not os.environ.get(name)
]

if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

records, _, _ = neo4j_driver.execute_query("RETURN 'connected' AS status")
records[0].data()


## 2. Create a Small Demo Knowledge Graph

This resets only the demo nodes labeled `Concept` and the demo relationships used below.


In [ ]:
reset_query = """
MATCH (n:Concept)
DETACH DELETE n
"""

seed_query = """
MERGE (llm:Concept {name: 'LLM'})
SET llm.description = 'Large language model used for generation and reasoning.'
MERGE (rag:Concept {name: 'RAG'})
SET rag.description = 'Retrieval-augmented generation that grounds answers with retrieved context.'
MERGE (kg:Concept {name: 'Knowledge Graph'})
SET kg.description = 'Graph structure of entities and relationships.'
MERGE (neo4j:Concept {name: 'Neo4j'})
SET neo4j.description = 'Graph database used to store and query connected data.'
MERGE (embed:Concept {name: 'Embeddings'})
SET embed.description = 'Vector representations used for semantic similarity.'
MERGE (vector:Concept {name: 'Vector Search'})
SET vector.description = 'Search over embeddings to retrieve semantically similar content.'
MERGE (chunking:Concept {name: 'Chunking'})
SET chunking.description = 'Splitting source text into smaller passages for retrieval.'

MERGE (rag)-[:USES]->(llm)
MERGE (rag)-[:USES]->(vector)
MERGE (vector)-[:USES]->(embed)
MERGE (rag)-[:USES]->(chunking)
MERGE (kg)-[:STORED_IN]->(neo4j)
MERGE (rag)-[:CAN_USE]->(kg)
MERGE (kg)-[:IMPROVES]->(rag)
MERGE (neo4j)-[:SUPPORTS]->(kg)
"""

neo4j_driver.execute_query(reset_query)
neo4j_driver.execute_query(seed_query)
print('Demo graph loaded.')


## 3. Run a Few Plain Cypher Queries


In [ ]:
query = """
MATCH (c:Concept)
RETURN c.name AS concept, c.description AS description
ORDER BY concept
"""

records, _, _ = neo4j_driver.execute_query(query)
pprint([record.data() for record in records])


In [ ]:
query = """
MATCH (source:Concept {name: 'RAG'})-[r]->(target:Concept)
RETURN type(r) AS relationship, target.name AS target
ORDER BY relationship, target
"""

records, _, _ = neo4j_driver.execute_query(query)
pprint([record.data() for record in records])


## 4. Inspect the Graph Schema

This uses `schema_utils.py` and typically works best when APOC is available in Neo4j.


In [ ]:
print(get_schema(neo4j_driver))


## 5. Optional: Ask a Question in Natural Language

If `OPENAI_API_KEY` is set, `Text2Cypher` can generate a Cypher query from a question.


In [ ]:
if not os.environ.get('OPENAI_API_KEY'):
    print('Skipping Text2Cypher because OPENAI_API_KEY is not set.')
else:
    from text2cypher import Text2Cypher

    question = 'Which concepts does RAG use? Return the concept names as target.'
    t2c = Text2Cypher(neo4j_driver)
    t2c.set_prompt_section('question', question)
    cypher = t2c.generate_cypher()
    print(cypher)
    records, _, _ = neo4j_driver.execute_query(cypher)
    pprint([record.data() for record in records])
